# Org1 Graph Construction

In [3]:
from pathlib import Path
import duckdb

In [4]:
ORG1_PATH = (
    Path("../data/organizations/organization=Org1")
    .resolve()
    .as_posix()
    + "/*.parquet"
)

print(ORG1_PATH)

C:/Users/Gulso/Documents/Gul/CredBlock-FL/data/organizations/organization=Org1/*.parquet


In [5]:
org1_users = duckdb.sql(f"""
    SELECT
        "User ID",
        ROW_NUMBER() OVER (
            ORDER BY "User ID"
        ) - 1 AS user_node_id

    FROM (
        SELECT DISTINCT
            "User ID"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    ORDER BY user_node_id
""").df()

print(org1_users.shape)
display(org1_users.head())

(791567, 2)


,User ID,user_node_id
0,-9223371191532286299,0
1,-9223351024921207274,1
2,-9223320431696609944,2
3,-9223311529942795212,3
4,-9223287066183308537,4


In [6]:
org1_ips = duckdb.sql(f"""
    SELECT
        "IP Address",
        ROW_NUMBER() OVER (
            ORDER BY "IP Address"
        ) - 1 AS ip_node_id

    FROM (
        SELECT DISTINCT
            "IP Address"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    ORDER BY ip_node_id
""").df()

print(org1_ips.shape)
display(org1_ips.head())

(733381, 2)


,IP Address,ip_node_id
0,10.0.0.1,0
1,10.0.0.10,1
2,10.0.0.100,2
3,10.0.0.101,3
4,10.0.0.102,4


In [7]:
USER_COUNT = len(org1_users)

org1_ips["global_ip_node_id"] = (
    org1_ips["ip_node_id"] + USER_COUNT
)

display(org1_ips.head())

print("User node ID range:")
print(org1_users["user_node_id"].min(), "to",
      org1_users["user_node_id"].max())

print("\nIP node ID range:")
print(org1_ips["global_ip_node_id"].min(), "to",
      org1_ips["global_ip_node_id"].max())

,IP Address,ip_node_id,global_ip_node_id
0,10.0.0.1,0,791567
1,10.0.0.10,1,791568
2,10.0.0.100,2,791569
3,10.0.0.101,3,791570
4,10.0.0.102,4,791571


User node ID range:
0 to 791566

IP node ID range:
791567 to 1524947


In [8]:
org1_edges = duckdb.sql(f"""
    WITH unique_pairs AS (
        SELECT DISTINCT
            "User ID",
            "IP Address"

        FROM read_parquet(
            '{ORG1_PATH}'
        )
    )

    SELECT
        u.user_node_id AS user_node_id,
        i.global_ip_node_id AS ip_node_id

    FROM unique_pairs p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_edges.shape)
display(org1_edges.head())

(1697608, 2)


,user_node_id,ip_node_id
0,780976,812976
1,211936,815654
2,436281,791693
3,293617,803542
4,211936,815810


In [9]:
print("Total edges:", len(org1_edges))

print("\nDuplicate edges:")
print(
    org1_edges.duplicated(
        subset=["user_node_id", "ip_node_id"]
    ).sum()
)

print("\nMissing values:")
print(org1_edges.isnull().sum())

print("\nUser node ID range in edges:")
print(
    org1_edges["user_node_id"].min(),
    "to",
    org1_edges["user_node_id"].max()
)

print("\nIP node ID range in edges:")
print(
    org1_edges["ip_node_id"].min(),
    "to",
    org1_edges["ip_node_id"].max()
)

print("\nUnique users represented in edges:")
print(org1_edges["user_node_id"].nunique())

print("\nUnique IPs represented in edges:")
print(org1_edges["ip_node_id"].nunique())

Total edges: 1697608

Duplicate edges:
0

Missing values:
user_node_id    0
ip_node_id      0
dtype: int64

User node ID range in edges:
0 to 791566

IP node ID range in edges:
791567 to 1524947

Unique users represented in edges:
791567

Unique IPs represented in edges:
733381


In [10]:
org1_edge_features = duckdb.sql(f"""
    WITH pair_features AS (
        SELECT
            "User ID",
            "IP Address",

            COUNT(*) AS total_interactions,

            SUM(
                CASE WHEN "Login Successful" = TRUE
                THEN 1 ELSE 0 END
            ) AS successful_interactions,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_interactions,

            AVG(
                CASE WHEN "Login Successful" = FALSE
                THEN 1.0 ELSE 0.0 END
            ) AS failure_ratio,

            MIN("Login Timestamp") AS first_seen,

            MAX("Login Timestamp") AS last_seen

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            "IP Address"
    )

    SELECT
        u.user_node_id,
        i.global_ip_node_id AS ip_node_id,

        p.total_interactions,
        p.successful_interactions,
        p.failed_interactions,
        p.failure_ratio,
        p.first_seen,
        p.last_seen

    FROM pair_features p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_edge_features.shape)
display(org1_edge_features.head())

(1697608, 8)


,user_node_id,ip_node_id,total_interactions,successful_interactions,failed_interactions,failure_ratio,first_seen,last_seen
0,620056,796178,1,1.0,0.0,0.000000,2020-02-04 07:35:38.590,2020-02-04 07:35:38.590
1,218471,1197474,14,12.0,2.0,0.142857,2020-02-04 07:41:12.925,2020-12-14 21:04:57.932
2,191950,806893,8,8.0,0.0,0.000000,2020-02-04 07:46:46.846,2020-11-23 11:02:00.988
3,308477,1220129,1,0.0,1.0,1.000000,2020-02-04 08:18:07.462,2020-02-04 08:18:07.462
4,616396,1405832,1,1.0,0.0,0.000000,2020-02-04 08:20:24.748,2020-02-04 08:20:24.748


In [11]:
print("Number of unique edges:")
print(len(org1_edge_features))

print("\nTotal interactions represented:")
print(org1_edge_features["total_interactions"].sum())

print("\nSuccessful interactions:")
print(org1_edge_features["successful_interactions"].sum())

print("\nFailed interactions:")
print(org1_edge_features["failed_interactions"].sum())

print("\nSuccess + Failed:")
print(
    org1_edge_features["successful_interactions"].sum()
    + org1_edge_features["failed_interactions"].sum()
)

print("\nMissing values:")
print(org1_edge_features.isnull().sum())

print("\nInvalid timestamp order:")
print(
    (
        org1_edge_features["first_seen"]
        > org1_edge_features["last_seen"]
    ).sum()
)

Number of unique edges:
1697608

Total interactions represented:
3080150

Successful interactions:
2219740.0

Failed interactions:
860410.0

Success + Failed:
3080150.0

Missing values:
user_node_id               0
ip_node_id                 0
total_interactions         0
successful_interactions    0
failed_interactions        0
failure_ratio              0
first_seen                 0
last_seen                  0
dtype: int64

Invalid timestamp order:
0


In [12]:
org1_temporal_edge_features = duckdb.sql(f"""
    WITH pair_windows AS (
        SELECT
            "User ID",
            "IP Address",

            time_bucket(
                INTERVAL '10 minutes',
                "Login Timestamp"
            ) AS ten_minute_window,

            COUNT(*) AS interactions_in_window,

            SUM(
                CASE WHEN "Login Successful" = FALSE
                THEN 1 ELSE 0 END
            ) AS failed_in_window

        FROM read_parquet(
            '{ORG1_PATH}'
        )

        GROUP BY
            "User ID",
            "IP Address",
            ten_minute_window
    ),

    pair_temporal AS (
        SELECT
            "User ID",
            "IP Address",

            COUNT(*) AS active_10min_windows,

            MAX(interactions_in_window)
                AS max_interactions_10min,

            MAX(failed_in_window)
                AS max_failed_interactions_10min

        FROM pair_windows

        GROUP BY
            "User ID",
            "IP Address"
    )

    SELECT
        u.user_node_id,
        i.global_ip_node_id AS ip_node_id,

        p.active_10min_windows,
        p.max_interactions_10min,
        p.max_failed_interactions_10min

    FROM pair_temporal p

    INNER JOIN org1_users u
        ON p."User ID" = u."User ID"

    INNER JOIN org1_ips i
        ON p."IP Address" = i."IP Address"
""").df()

print(org1_temporal_edge_features.shape)
display(org1_temporal_edge_features.head())

(1697608, 5)


,user_node_id,ip_node_id,active_10min_windows,max_interactions_10min,max_failed_interactions_10min
0,147113,1330086,2,1,0.0
1,16649,1329994,14,1,1.0
2,774095,991271,1,1,1.0
3,424829,1398997,38,2,1.0
4,480786,800502,1,1,0.0
